# Continue training (epochs 300 to 600), pick the checkpoint, final evaluation

Continues fine-tuning to 600 epochs, compares checkpoints 300/400/500/600 with the new sampler, evaluates the best one (epoch 400) on 10,000 images, and scores the reference model with the same metrics.

In [1]:
!pip install -q torch-fidelity diffusers

In [2]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
RUN       = "G_finetune"; ARCH = "pretrained"; SCHEDULE = "linear"
SWEEP_N   = 2000
SAMPLERS  = [dict(steps=100, eta=0.0), dict(steps=100, eta=1.0),
             dict(steps=250, eta=0.0), dict(steps=250, eta=1.0)]
BEST_SAMPLER = dict(steps=250, eta=1.0)
CKPTS     = ["weights_ep200.pt", "weights_ep250.pt", "model.pt", "ema.pt"]
CKPT_SWEEP_STEPS = 100
FINAL_N   = 10000
BS        = 250
SEED      = 0

EXTRA_EPOCHS = 300
SAVE_EVERY   = 100
LR, BATCH, WARMUP, EMA_DECAY, CLIP = 5e-5, 32, 500, 0.9995, 1.0
STEP0 = 40500

In [3]:
import os, sys, glob, json, math, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, torch, torchvision, kagglehub
import torchvision.transforms as T
from torchvision.utils import save_image
from tqdm.auto import tqdm
from diffusers import UNet2DModel
for m in [k for k in list(sys.modules) if k.startswith("torchmetrics")]:
    del sys.modules[m]
import torch_fidelity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.inception import InceptionScore

assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2"
DEVS = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
DEV = DEVS[0]
OUT = "/kaggle/working/G_continue"; os.makedirs(OUT, exist_ok=True)
print("GPUs:", [torch.cuda.get_device_name(d) for d in DEVS])

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return hits[0] if hits else None

assert find(f"{RUN}/ema.pt"), "Attach the G training notebook (Add Input -> Notebooks -> ddpm-task)."
print("G weights dir:", os.path.dirname(find(f"{RUN}/ema.pt")))
print("baseline weights:", find("baseline_asis/model.pt"))

hits = sorted(glob.glob("/kaggle/input/**/flowers-recognition", recursive=True))
path = hits[0] if hits else kagglehub.dataset_download("alxmamaev/flowers-recognition")
tf = T.Compose([T.ToTensor(), T.Resize((32, 32), interpolation=T.InterpolationMode.BICUBIC, antialias=True)])
ds = torchvision.datasets.ImageFolder(root=path, transform=tf)
X_float = torch.stack([ds[i][0] for i in range(len(ds))])
real_u8 = (X_float.clamp(0, 1) * 255).round().to(torch.uint8)
print("real:", tuple(real_u8.shape))

TSTEPS = 1000
beta = torch.linspace(1e-4, 0.02, TSTEPS, dtype=torch.float32)
a_bar = torch.cumprod(1 - beta, 0)

def build_model(arch):
    if arch == "pretrained":
        return UNet2DModel.from_pretrained("google/ddpm-cifar10-32")
    down = ("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D")
    up   = ("AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D")
    return UNet2DModel(in_channels=3, out_channels=3, block_out_channels=[64, 128, 256, 256],
                       layers_per_block=2, down_block_types=down, up_block_types=up)

def load_nets(weights_path, arch):
    sd = torch.load(weights_path, map_location="cpu"); nets = []
    for d in DEVS:
        m = build_model(arch); m.load_state_dict(sd); nets.append(m.to(d).eval())
    return nets

GPUs: ['Tesla T4', 'Tesla T4']
G weights dir: /kaggle/input/notebooks/malharinamdar/ddpm-task/G_finetune
baseline weights: /kaggle/input/notebooks/malharinamdar/notebook11174aea07/baseline_asis/model.pt
real: (4317, 3, 32, 32)


In [4]:
import copy, math
import torch.nn.functional as F
wdir = os.path.dirname(find(f"{RUN}/ema.pt"))
X_train = (X_float * 2 - 1).to(DEV); N = len(X_train)
sqrt_ab_d, sqrt_1m_ab_d = a_bar.sqrt().to(DEV), (1 - a_bar).sqrt().to(DEV)
model = build_model(ARCH); model.load_state_dict(torch.load(f"{wdir}/model.pt", map_location="cpu")); model.to(DEV)
ema = build_model(ARCH); ema.load_state_dict(torch.load(f"{wdir}/ema.pt", map_location="cpu")); ema.to(DEV).eval().requires_grad_(False)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def ema_update(step):
    d = min(EMA_DECAY, (1 + step) / (10 + step))
    for pe, p in zip(ema.parameters(), model.parameters()): pe.lerp_(p, 1 - d)
    for be, b in zip(ema.buffers(), model.buffers()): be.copy_(b)

torch.manual_seed(SEED + 7)
log, step, t0 = [], 0, time.time()
for epoch in range(1, EXTRA_EPOCHS + 1):
    model.train(); perm = torch.randperm(N, device=DEV); losses, nan_steps = [], 0
    for i in range(0, N, BATCH):
        x0 = X_train[perm[i:i + BATCH]]
        x0 = torch.where(torch.rand(len(x0), 1, 1, 1, device=DEV) < 0.5, x0.flip(3), x0)
        t = torch.randint(1, TSTEPS, (len(x0),), device=DEV)
        noise = torch.randn_like(x0)
        xt = sqrt_ab_d[t].view(-1, 1, 1, 1) * x0 + sqrt_1m_ab_d[t].view(-1, 1, 1, 1) * noise
        for g in opt.param_groups: g["lr"] = LR * min(1.0, (step + 1) / WARMUP)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = F.mse_loss(model(xt, t).sample.float(), noise)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scaler.step(opt); scaler.update(); ema_update(STEP0 + step)
        lv = loss.item()
        if math.isfinite(lv): losses.append(lv)
        else: nan_steps += 1
        step += 1
    row = dict(epoch=300 + epoch, loss=float(np.mean(losses)) if losses else float("nan"), nan_steps=nan_steps,
               minutes=round((time.time() - t0) / 60, 2))
    log.append(row); print(row)
    if epoch % SAVE_EVERY == 0:
        torch.save(ema.state_dict(), f"{OUT}/cont_ep{300 + epoch}.pt")
    json.dump(log, open(f"{OUT}/cont_train_log.json", "w"), indent=2)
torch.save(model.state_dict(), f"{OUT}/cont_model.pt")
train_minutes = (time.time() - t0) / 60
print(f"continued training done: {train_minutes:.1f} min, NaN steps: {sum(r['nan_steps'] for r in log)}")
del model, ema, opt, scaler, X_train; import gc; gc.collect(); torch.cuda.empty_cache()

{'epoch': 301, 'loss': 0.038008507479120185, 'nan_steps': 0, 'minutes': 0.4}
{'epoch': 302, 'loss': 0.04116523487286435, 'nan_steps': 0, 'minutes': 0.75}
{'epoch': 303, 'loss': 0.03839362031193795, 'nan_steps': 0, 'minutes': 1.11}
{'epoch': 304, 'loss': 0.03923673538698091, 'nan_steps': 0, 'minutes': 1.47}
{'epoch': 305, 'loss': 0.03916360604128352, 'nan_steps': 0, 'minutes': 1.84}
{'epoch': 306, 'loss': 0.04015989727168171, 'nan_steps': 0, 'minutes': 2.21}
{'epoch': 307, 'loss': 0.03868064148420537, 'nan_steps': 0, 'minutes': 2.58}
{'epoch': 308, 'loss': 0.0393562481497173, 'nan_steps': 0, 'minutes': 2.95}
{'epoch': 309, 'loss': 0.03801586603814805, 'nan_steps': 0, 'minutes': 3.32}
{'epoch': 310, 'loss': 0.039176530549647634, 'nan_steps': 0, 'minutes': 3.69}
{'epoch': 311, 'loss': 0.039677693887993144, 'nan_steps': 0, 'minutes': 4.06}
{'epoch': 312, 'loss': 0.0396507471523903, 'nan_steps': 0, 'minutes': 4.43}
{'epoch': 313, 'loss': 0.03936324674912073, 'nan_steps': 0, 'minutes': 4.8}


In [5]:
def taus_for(steps):
    return np.linspace(0, TSTEPS - 1, steps).round().astype(int)[::-1].copy()

@torch.no_grad()
def sample_on(i, nets, n, seed, bs, steps, eta, tag=None):
    d, m = DEVS[i], nets[i]; ab = a_bar.to(d); taus = taus_for(steps)
    g = torch.Generator(device=d).manual_seed(seed * 1000 + i); outs = []
    for s in range(0, n, bs):
        b = min(bs, n - s)
        x = torch.randn((b, 3, 32, 32), device=d, generator=g)
        for k, t in enumerate(tqdm(taus, desc=f"gpu{i} {steps}st eta{eta} batch {s//bs+1}/{(n+bs-1)//bs}", position=i, leave=False)):
            a_t = ab[t]
            a_s = ab[taus[k + 1]] if k + 1 < len(taus) else torch.ones((), device=d)
            eps = m(x, torch.full((b,), int(t), dtype=torch.long, device=d)).sample
            x0 = ((x - (1 - a_t).sqrt() * eps) / a_t.sqrt()).clamp(-1, 1)
            eps = (x - a_t.sqrt() * x0) / (1 - a_t).sqrt()
            sigma = eta * ((1 - a_s) / (1 - a_t)).sqrt() * (1 - a_t / a_s).sqrt()
            x = a_s.sqrt() * x0 + (1 - a_s - sigma ** 2).clamp(min=0).sqrt() * eps
            if eta > 0 and k + 1 < len(taus):
                x = x + sigma * torch.randn(x.shape, device=d, generator=g)
        outs.append(((x.clamp(-1, 1) + 1) / 2 * 255).round().to(torch.uint8).cpu())
        if tag: np.save(f"{OUT}/{tag}_gpu{i}.npy", torch.cat(outs).numpy())
    return torch.cat(outs)

def sample(nets, n, seed, bs, steps, eta, tag=None):
    k = len(DEVS); shares = [n // k + (1 if i < n % k else 0) for i in range(k)]
    with ThreadPoolExecutor(k) as ex:
        parts = list(ex.map(lambda i: sample_on(i, nets, shares[i], seed, bs, steps, eta, tag), range(k)))
    return torch.cat(parts)

@torch.no_grad()
def fid_kid(a_u8, b_u8):
    torch.cuda.empty_cache()
    fid = FrechetInceptionDistance(feature=2048).to(DEV)
    kid = KernelInceptionDistance(subset_size=min(1000, len(a_u8), len(b_u8))).to(DEV)
    for x, flag in ((a_u8, True), (b_u8, False)):
        for i in range(0, len(x), 100):
            xb = x[i:i + 100].to(DEV); fid.update(xb, real=flag); kid.update(xb, real=flag)
    km, ks = kid.compute(); out = (fid.compute().item(), km.item(), ks.item())
    del fid, kid; torch.cuda.empty_cache()
    return out

sweep = []
def record(row):
    sweep.append(row); print(row)
    json.dump(sweep, open(f"{OUT}/sweep.json", "w"), indent=2)

In [6]:
wdir = os.path.dirname(find(f"{RUN}/ema.pt"))
nets = load_nets(f"{wdir}/ema.pt", ARCH)
for cfg in ([] if BEST_SAMPLER else SAMPLERS):
    t0 = time.time()
    gen = sample(nets, SWEEP_N, SEED, BS, cfg["steps"], cfg["eta"])
    f, km, ks = fid_kid(real_u8, gen)
    record(dict(stage="A_sampler", ckpt="ema.pt", **cfg, fid_2k=f, kid_2k=km, minutes=round((time.time() - t0) / 60, 1)))
best_cfg = BEST_SAMPLER or min([r for r in sweep if r["stage"] == "A_sampler"], key=lambda r: r["fid_2k"])
print("best sampler:", best_cfg)

best sampler: {'steps': 250, 'eta': 1.0}


In [7]:
del nets; torch.cuda.empty_cache()
CANDIDATES = [("ep300", f"{wdir}/ema.pt")] + [(f"ep{300 + e}", f"{OUT}/cont_ep{300 + e}.pt")
                                              for e in range(SAVE_EVERY, EXTRA_EPOCHS + 1, SAVE_EVERY)]
for name, p in CANDIDATES:
    if not os.path.exists(p): print("skip (missing):", name); continue
    t0 = time.time(); nets = load_nets(p, ARCH)
    gen = sample(nets, SWEEP_N, SEED, BS, CKPT_SWEEP_STEPS, best_cfg["eta"])
    f, km, ks = fid_kid(real_u8, gen)
    record(dict(stage="B_ckpt", ckpt=name, path=p, steps=CKPT_SWEEP_STEPS, eta=best_cfg["eta"], fid_2k=f, kid_2k=km,
                minutes=round((time.time() - t0) / 60, 1)))
    del nets; torch.cuda.empty_cache()
best_row = min([r for r in sweep if r["stage"] == "B_ckpt"], key=lambda r: r["fid_2k"])
best_ck, best_path = best_row["ckpt"], best_row["path"]
print("best checkpoint:", best_ck, "| sampler:", best_cfg["steps"], "steps, eta", best_cfg["eta"])

{'stage': 'B_ckpt', 'ckpt': 'ep300', 'path': '/kaggle/input/notebooks/malharinamdar/ddpm-task/G_finetune/ema.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 28.51896095275879, 'kid_2k': 0.006705126725137234, 'minutes': 5.1}


{'stage': 'B_ckpt', 'ckpt': 'ep400', 'path': '/kaggle/working/G_continue/cont_ep400.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 28.158124923706055, 'kid_2k': 0.006651353556662798, 'minutes': 5.3}


{'stage': 'B_ckpt', 'ckpt': 'ep500', 'path': '/kaggle/working/G_continue/cont_ep500.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 30.510093688964844, 'kid_2k': 0.00841632392257452, 'minutes': 5.4}


{'stage': 'B_ckpt', 'ckpt': 'ep600', 'path': '/kaggle/working/G_continue/cont_ep600.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 30.23662567138672, 'kid_2k': 0.00843946635723114, 'minutes': 5.4}
best checkpoint: ep400 | sampler: 250 steps, eta 1.0


In [8]:
nets = load_nets(best_path, ARCH)
t0 = time.time()
gen = sample(nets, FINAL_N, SEED + 1, BS, best_cfg["steps"], best_cfg["eta"], tag="final")
final_minutes = (time.time() - t0) / 60
np.save(f"{OUT}/final_samples.npy", gen.numpy())
save_image(gen[:64].float() / 255, f"{OUT}/final_samples_grid.png", nrow=8)
del nets; torch.cuda.empty_cache()

res = dict(ckpt=best_ck, steps=best_cfg["steps"], eta=best_cfg["eta"], n_total=len(gen), sample_minutes=round(final_minutes, 1))
r2k = [fid_kid(real_u8, gen[j:j + 2000]) for j in range(0, len(gen) - 1999, 2000)]
res["fid_2k"] = [round(r[0], 3) for r in r2k]; res["kid_2k"] = [r[1] for r in r2k]
res["fid_2k_mean"], res["fid_2k_std"] = float(np.mean(res["fid_2k"])), float(np.std(res["fid_2k"]))
res["kid_2k_mean"], res["kid_2k_std"] = float(np.mean(res["kid_2k"])), float(np.std(res["kid_2k"]))
Ns = [n for n in (2000, 4000, 6000, 8000, 10000) if n <= len(gen)]; fid_at = {}
for n in Ns:
    vals = [fid_kid(real_u8, gen[torch.randperm(len(gen), generator=torch.Generator().manual_seed(r))[:n]])[0]
            for r in range(3 if n < len(gen) else 1)]
    fid_at[n] = float(np.mean(vals)); print(f"FID@{n}: {fid_at[n]:.3f}")
res["fid_at_N"] = fid_at
b, a = np.polyfit([1 / n for n in Ns], [fid_at[n] for n in Ns], 1); res["fid_inf"], res["fid_inf_slope"] = float(a), float(b)
json.dump(res, open(f"{OUT}/final_results.json", "w"), indent=2); print("core final results saved:", res["fid_2k_mean"], fid_at.get(10000), res["fid_inf"])

def inception_score(x_u8):
    m = InceptionScore().to(DEV)
    for i in range(0, len(x_u8), 100): m.update(x_u8[i:i + 100].to(DEV))
    mu, sd = m.compute(); return mu.item(), sd.item()
class U8(torch.utils.data.Dataset):
    def __init__(self, x): self.x = x
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i]
def prc(x_u8):
    r = torch_fidelity.calculate_metrics(input1=U8(x_u8[:2000]), input2=U8(real_u8), prc=True, cuda=True, batch_size=100, verbose=False)
    return r["precision"], r["recall"]
def nn_check(x_u8, fname):
    R = real_u8.float().flatten(1) / 255; G = x_u8[:64].float().flatten(1) / 255; exact = "donot_use_mm_for_euclid_dist"
    d_gen, idx = torch.cdist(G, R, compute_mode=exact).min(1)
    q = torch.randperm(len(R), generator=torch.Generator().manual_seed(0))[:64]
    dr = torch.cdist(R[q], R, compute_mode=exact); dr[torch.arange(64), q] = float("inf")
    save_image(torch.stack([x_u8[:64], real_u8[idx]], 1).flatten(0, 1)[:32].float() / 255, f"{OUT}/{fname}", nrow=8)
    return d_gen.mean().item(), dr.min(1).values.mean().item()

try: res["is_mean"], res["is_std"] = inception_score(gen)
except Exception as e: res["is_error"] = repr(e); print("IS failed:", e)
try: res["precision"], res["recall"] = prc(gen)
except Exception as e: res["prc_error"] = repr(e); print("precision/recall failed:", e)
res["nn_dist_gen"], res["nn_dist_real"] = nn_check(gen, "final_nearest_neighbours.png")
json.dump(res, open(f"{OUT}/final_results.json", "w"), indent=2)
print(json.dumps(res, indent=2))

FID@2000: 26.816
FID@4000: 21.055
FID@6000: 19.207
FID@8000: 18.201
FID@10000: 17.698
core final results saved: 27.224 17.697668075561523 15.376818545055151


{
  "ckpt": "ep400",
  "steps": 250,
  "eta": 1.0,
  "n_total": 10000,
  "sample_minutes": 59.1,
  "fid_2k": [
    27.339,
    27.132,
    26.639,
    27.215,
    27.795
  ],
  "kid_2k": [
    0.005946828983724117,
    0.005598821677267551,
    0.005874068476259708,
    0.00586184486746788,
    0.006048311945050955
  ],
  "fid_2k_mean": 27.224,
  "fid_2k_std": 0.3714932031679724,
  "kid_2k_mean": 0.005865975189954043,
  "kid_2k_std": 0.0001491560179235699,
  "fid_at_N": {
    "2000": 26.81639035542806,
    "4000": 21.055336634318035,
    "6000": 19.207083384195965,
    "8000": 18.201011021931965,
    "10000": 17.697668075561523
  },
  "fid_inf": 15.376818545055151,
  "fid_inf_slope": 22855.5299966363,
  "is_mean": 5.540163516998291,
  "is_std": 0.12839311361312866,
  "precision": 0.8050000071525574,
  "recall": 0.5232800841331482,
  "nn_dist_gen": 10.476552963256836,
  "nn_dist_real": 11.142486572265625
}


In [9]:
def safe(fn):
    try: return fn()
    except Exception as e: print("failed:", e); return repr(e)
base = {}
bw = find("baseline_asis/model.pt"); bs_npy = find("baseline_asis/samples_seed0.npy")
base["is_real"] = safe(lambda: inception_score(real_u8))
if bs_npy:
    g0 = torch.from_numpy(np.load(bs_npy))
    base["orig_sampler"] = dict(fid_kid=safe(lambda: fid_kid(real_u8, g0)), is_=safe(lambda: inception_score(g0)),
                                prc=safe(lambda: prc(g0)))
    json.dump(base, open(f"{OUT}/baseline_results.json", "w"), indent=2)
if bw:
    try:
        nets = load_nets(bw, "orig")
        g1 = sample(nets, SWEEP_N, SEED, BS, best_cfg["steps"], best_cfg["eta"])
        save_image(g1[:64].float() / 255, f"{OUT}/baseline_best_sampler_grid.png", nrow=8)
        del nets; torch.cuda.empty_cache()
        base["best_sampler"] = dict(steps=best_cfg["steps"], eta=best_cfg["eta"], fid_kid=safe(lambda: fid_kid(real_u8, g1)),
                                    is_=safe(lambda: inception_score(g1)), prc=safe(lambda: prc(g1)))
    except Exception as e:
        base["best_sampler_error"] = repr(e); print("baseline resample failed:", e)
json.dump(base, open(f"{OUT}/baseline_results.json", "w"), indent=2)
print(json.dumps(base, indent=2))
print("\nSWEEP SUMMARY"); [print(r) for r in sweep]

{
  "is_real": [
    6.290989398956299,
    0.3087352514266968
  ],
  "orig_sampler": {
    "fid_kid": [
      248.54388427734375,
      0.1681000292301178,
      0.0038599297404289246
    ],
    "is_": [
      3.8377187252044678,
      0.21492362022399902
    ],
    "prc": [
      0.8149999976158142,
      0.01714153401553631
    ]
  },
  "best_sampler": {
    "steps": 250,
    "eta": 1.0,
    "fid_kid": [
      172.22982788085938,
      0.12639807164669037,
      0.0023198751732707024
    ],
    "is_": [
      2.718881368637085,
      0.12488572299480438
    ],
    "prc": [
      0.840499997138977,
      0.010423905216157436
    ]
  }
}
SWEEP SUMMARY
{'stage': 'B_ckpt', 'ckpt': 'ep300', 'path': '/kaggle/input/notebooks/malharinamdar/ddpm-task/G_finetune/ema.pt', 'steps': 100, 'eta': 1.0, 'fid_2k': 28.51896095275879, 'kid_2k': 0.006705126725137234, 'minutes': 5.1}
{'stage': 'B_ckpt', 'ckpt': 'ep400', 'path': '/kaggle/working/G_continue/cont_ep400.pt', 'steps': 100, 'eta': 1.0, 'fid_2k

[None, None, None, None]